In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_csv('../data/Telco-Customer-Churn-final.csv')
print(f"Loaded dataset: {df.shape[0]} rows x {df.shape[1]} columns")
print(f"\nChurn distribution:\n{df['Churn'].value_counts()}")
print(f"\nChurn ratio: {df['Churn'].value_counts(normalize=True).round(3).to_dict()}")

## Step 7.1: Train-Test Split

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop('Churn', axis=1)
y = df['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Confirm stratification worked correctly
print(f"Training rows : {len(X_train)}")
print(f"Test rows     : {len(X_test)}")
print()
print("Churn ratio — full dataset:")
print(y.value_counts(normalize=True).round(3))
print()
print("Churn ratio — training set:")
print(y_train.value_counts(normalize=True).round(3))
print()
print("Churn ratio — test set:")
print(y_test.value_counts(normalize=True).round(3))

## Step 7.2: Handling Class Imbalance with SMOTE

SMOTE is applied **only to the training set** — the test set must remain untouched to reflect real-world class ratios.

In [ ]:
from imblearn.over_sampling import SMOTE

# Show counts BEFORE SMOTE
print("BEFORE SMOTE (training set):")
print(y_train.value_counts())

smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

# Show counts AFTER SMOTE
print("\nAFTER SMOTE (training set):")
print(pd.Series(y_train_resampled).value_counts())

print("\nTEST SET (untouched — real-world ratio preserved):")
print(y_test.value_counts())

synthetic = pd.Series(y_train_resampled).value_counts()[1] - y_train.value_counts()[1]
print(f"\nSynthetic churn examples generated by SMOTE: {synthetic}")

## Save All 4 Splits to Disk

Saving so the next notebook (modeling) can load directly — no need to repeat splitting or SMOTE.

In [ ]:
import os

os.makedirs('../data/splits', exist_ok=True)

pd.DataFrame(X_train_resampled, columns=X.columns).to_csv('../data/splits/X_train.csv', index=False)
pd.Series(y_train_resampled, name='Churn').to_csv('../data/splits/y_train.csv', index=False)
X_test.to_csv('../data/splits/X_test.csv', index=False)
y_test.to_csv('../data/splits/y_test.csv', index=False)

print("Saved to ../data/splits/:")
print(f"  X_train.csv  — {X_train_resampled.shape[0]} rows x {X_train_resampled.shape[1]} cols (SMOTE-balanced)")
print(f"  y_train.csv  — {len(y_train_resampled)} labels")
print(f"  X_test.csv   — {X_test.shape[0]} rows x {X_test.shape[1]} cols")
print(f"  y_test.csv   — {len(y_test)} labels (real-world ratio preserved)")